
# Built-in GMLS Interpolation Verification (numpy/scipy Backend)
In this example, we use an analytical function
to test and verify MatCal's built-in numpy/scipy GMLS
implementation by explicitly selecting the ``"scipy"`` backend.

The built-in implementation uses the same radius-based neighbor
search and local polynomial least-squares approach as pycompadre,
but relies only on numpy, scipy, and their ``cKDTree`` and sparse
matrix utilities.  This example follows the same verification
procedure as `sphx_glr_full_field_verification_examples_plot_a_interpolation_methods_verification.py`
to demonstrate that the scipy backend produces equivalent
results.

The verification procedure:

#.  We evaluate an analytical function on a set of points over a
    domain that is 5% smaller than the domain we
    will interpolate and extrapolate to.  This will
    be referred to as our measurement grid and is
    meant to be representative of experimental data.
#.  We add noise with a normal distribution to the
    data generated in the previous step.  The noise
    has a maximum amplitude of 2.5% of the function
    maximum value to represent noise present
    in measured data.
#.  We create a separate domain with 75% of the points
    from the measured grid that is 5% larger
    in both the X and Y directions and evaluate the function
    at these points without noise.  This is
    to be used as the truth
    value of the function and this set of
    points will be referred to as the simulation
    grid.  We will attempt to reproduce
    these values with our built-in GMLS
    interpolation and extrapolation.
#.  We loop over different input options to the GMLS
    algorithm and evaluate the accuracy of the method
    against the truth data with three measures of error:
    (1) the maximum percent error of the field produced
    by the GMLS tool, (2) the normalized L2 norm of this
    field, and (3) plots of the error field for all of the
    input options studied.

To begin we import the libraries and tools we will be using.


In [ ]:
# sphinx_gallery_thumbnail_number = 2
from matcal import *
from matcal.full_field.field_mappers import (
    _build_gmls_weight_matrix,
)
import numpy as np
import matplotlib.pyplot as plt

## Define measurement domain
The domain is about 152 mm high (6 inches) and 76 mm wide (3 inches).
The measured grid has 400 points in each dimension (x, y).



In [ ]:
H = 6 * 0.0254
W = 3 * 0.0254

measured_num_points = 400
measured_xs = np.linspace(-W / 2, W / 2, measured_num_points)
measured_ys = np.linspace(-H / 2, H / 2, measured_num_points)

measured_x_grid, measured_y_grid = np.meshgrid(measured_xs, measured_ys)

## Define the test function
We use the same analytical function as in the pycompadre
verification example: an additive combination of sinusoids
and a linear function multiplied by a smooth approximation
to a Dirac delta.



In [ ]:
def analytical_function(X, Y):
    small = H / 20
    func = (
        H / 5 * np.sin(np.pi * Y / 2 / (H / 2))
        - W / 50 * X / (W / 2)
        + H / 40 * np.sin(np.pi * Y / 2 / (H / 20))
        + W / 100 * np.sin(X / (W / 20))
    ) * (1 + small / (np.pi * (X**2 + Y**2 + small**2)))
    return func

## Evaluate the function and add noise



In [ ]:
measured_func = analytical_function(measured_x_grid, measured_y_grid)
rng = np.random.default_rng(42)
noise_amp = 0.025 * np.max(measured_func)
noise = (
    rng.random((measured_num_points, measured_num_points)) * noise_amp
    - noise_amp / 2
)
measured_func += noise

from matplotlib import cm

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.plot_surface(
    measured_x_grid, measured_y_grid, measured_func, cmap=cm.coolwarm
)
plt.xlabel("X")
plt.ylabel("Y")
ax.set_zlabel("Z")
plt.title("Measured Data (with noise)")
plt.show()

## Create simulation grid and truth data



In [ ]:
sim_num_points = 300
sim_xs = np.linspace(-W / 2 * 1.025, W / 2 * 1.025, sim_num_points)
sim_ys = np.linspace(-H / 2 * 1.025, H / 2 * 1.025, sim_num_points)

sim_x_grid, sim_y_grid = np.meshgrid(sim_xs, sim_ys)
sim_truth_func = analytical_function(sim_x_grid, sim_y_grid)

## Prepare data for MatCal's mapping tools



In [ ]:
measured_dict = {
    "x": measured_x_grid.reshape(measured_num_points**2),
    "y": measured_y_grid.reshape(measured_num_points**2),
    "val": measured_func.reshape(1, measured_num_points**2),
}
measured_data = convert_dictionary_to_field_data(
    measured_dict, coordinate_names=["x", "y"]
)

sim_truth_dict = {
    "x": sim_x_grid.reshape(sim_num_points**2),
    "y": sim_y_grid.reshape(sim_num_points**2),
    "val": sim_truth_func.reshape(1, sim_num_points**2),
}
sim_truth_data = convert_dictionary_to_field_data(
    sim_truth_dict, coordinate_names=["x", "y"]
)

## Parameter study setup
We study polynomial orders 1 through 3 with search radius
multipliers from 1.5 to 4.0 (plus 5.0).



In [ ]:
polynomial_orders = [1, 2, 3]
search_radius_mults = list(np.linspace(1.5, 4, 11))
search_radius_mults.append(5.0)

## Run the parameter study using ``meshless_remapping``
Here we call :func:`~matcal.full_field.field_mappers.meshless_remapping`
with ``backend="scipy"`` to ensure the built-in numpy/scipy GMLS
implementation is used, even when pycompadre is installed.  We also
demonstrate using the low-level
:func:`~matcal.full_field.field_mappers._build_gmls_weight_matrix`
helper at the end of this example to show the sparse weight matrix
directly.

Error measures:

\begin{align}e_{norm} = 100\frac{\lVert f^h_s-f_s\rVert_2}{m^2\max\left(f_s\right)}\end{align}

\begin{align}e_{max} = 100\frac{\lVert f^h_s-f_s\rVert_{\infty}}{\max\left(f_s\right)}\end{align}



In [ ]:
normalization_constant = np.max(sim_truth_func)

error_fields = []
error_norms = []
error_maxes = []
for poly_order in polynomial_orders:
    error_fields_by_search_rad = []
    error_norms_by_search_rad = []
    error_maxes_by_search_rad = []
    for search_rad_mult in search_radius_mults:
        mapped_data = meshless_remapping(
            measured_data,
            ["val"],
            sim_truth_data.spatial_coords,
            poly_order,
            search_rad_mult,
            backend="scipy",
        )
        error_field = mapped_data["val"] - sim_truth_data["val"]
        error_fields_by_search_rad.append(error_field)
        error_norm = (
            np.linalg.norm(error_field)
            / sim_num_points**2
            * 100
            / normalization_constant
        )
        error_norms_by_search_rad.append(error_norm)
        error_max = (
            np.max(np.abs(error_field)) / normalization_constant * 100
        )
        error_maxes_by_search_rad.append(error_max)
    error_fields.append(error_fields_by_search_rad)
    error_norms.append(error_norms_by_search_rad)
    error_maxes.append(error_maxes_by_search_rad)

error_fields = np.array(error_fields)
error_norms = np.array(error_norms)
error_maxes = np.array(error_maxes)

## Visualize error measures as heatmaps



In [ ]:
from seaborn import heatmap
import matplotlib.colors as colors

search_rad_mult_labels = [f"{i:.2f}" for i in search_radius_mults]
plt.figure("$e_{{norm}}$ (scipy backend)", figsize=(6, 4),
           constrained_layout=True)
heatmap(
    error_norms.T,
    annot=True,
    norm=colors.LogNorm(),
    xticklabels=polynomial_orders,
    yticklabels=search_rad_mult_labels,
)
plt.xlabel("polynomial order")
plt.ylabel("search radius multiplier")
plt.title("$e_{{norm}}$ — Built-in scipy GMLS")

plt.figure("$e_{{max}}$ (scipy backend)", figsize=(6, 4),
           constrained_layout=True)
heatmap(
    error_maxes.T,
    annot=True,
    norm=colors.LogNorm(),
    xticklabels=polynomial_orders,
    yticklabels=search_rad_mult_labels,
)
plt.xlabel("polynomial order")
plt.ylabel("search radius multiplier")
plt.title("$e_{{max}}$ — Built-in scipy GMLS")
plt.show()

From these heatmaps, we observe the same trends as the pycompadre
verification:

- Linear polynomials (order 1) produce the lowest maximum errors,
  particularly important when extrapolation is present.
- Higher search radius multipliers increase smoothing and reduce
  noise sensitivity.
- The built-in scipy backend reproduces the same accuracy patterns
  as pycompadre for all tested parameter combinations; however, 
  errors are higher in the higher polynomial/lower radius regions.



## Visualize error fields
We plot the absolute percent error fields for each parameter
combination to visualize spatial error distribution.



In [ ]:
num_polys = len(polynomial_orders)
num_radiis = len(search_radius_mults)
max_noise_error = noise_amp / 2 * 100 / normalization_constant

fig = plt.figure(
    "error fields — scipy GMLS backend",
    figsize=(5 * num_polys, 5 * num_radiis),
    constrained_layout=True,
)
for row in range(num_polys):
    for col in range(num_radiis):
        ax = plt.subplot(
            num_radiis, num_polys, (row + 1) + num_polys * col
        )
        error_field = np.abs(
            error_fields[row, col].reshape(
                sim_num_points, sim_num_points
            )
            / normalization_constant
            * 100
        )
        levs = list(np.linspace(0, max_noise_error, 6))
        max_err = np.max(error_field)
        if max_err > max_noise_error * 3:
            levs += [max_err / 2, max_err]
        elif max_err > max_noise_error:
            levs += [max_err]
        cs = ax.contourf(
            sim_x_grid,
            sim_y_grid,
            error_field,
            levs,
            norm=colors.PowerNorm(gamma=0.3),
            cmap="magma",
        )
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.title(
            f"polynomial order {row + 1}\nsearch "
            f"radius multiplier {search_radius_mults[col]:1.2f}"
        )
        plt.colorbar(cs, ax=ax)
plt.show()

## Observations
The error field patterns confirm:

#. Extrapolation error is highest at domain boundaries and
   increases with polynomial order — identical to the pycompadre
   behavior.
#. Noise filtering increases with search radius multiplier
   for all polynomial orders.
#. Higher-order polynomials capture smooth features better
   at larger search radii but produce larger edge artifacts.
#. The built-in numpy/scipy GMLS produces results that are
   qualitatively and quantitatively consistent with pycompadre
   for this verification problem.

These results validate that the built-in scipy backend is a
reliable alternative when pycompadre is not available, and that
the same default parameters (polynomial order 1, search radius
multiplier 2.75) provide a good balance between speed and
accuracy for both interpolation and extrapolation tasks.



## Direct sparse weight matrix demonstration
For users interested in the low-level mechanics, we demonstrate
building the sparse weight matrix directly using
:func:`~matcal.full_field.field_mappers._build_gmls_weight_matrix`.
This shows the sparse structure of the interpolation operator.



In [ ]:
from scipy import sparse

source_coords = measured_data.spatial_coords
target_coords = sim_truth_data.spatial_coords

# Build weight matrix for order 1, epsilon multiplier 2.75
W_matrix = _build_gmls_weight_matrix(
    source_coords, target_coords,
    polynomial_order=1, epsilon_multiplier=2.75
)

print(f"Weight matrix shape: {W_matrix.shape}")
print(f"Number of non-zeros: {W_matrix.nnz}")
print(f"Sparsity: {1 - W_matrix.nnz / (W_matrix.shape[0] * W_matrix.shape[1]):.6f}")
print(f"Average non-zeros per target: {W_matrix.nnz / W_matrix.shape[0]:.1f}")

# Apply the weight matrix to get mapped values
source_vals = measured_data["val"].ravel()
mapped_vals = W_matrix @ source_vals
truth_vals = sim_truth_data["val"].ravel()

direct_error = np.max(np.abs(mapped_vals - truth_vals)) / normalization_constant * 100
print(f"Max percent error (direct W @ f): {direct_error:.4f}%")